# Cosmos 3 Edge for First-Timers: The Settings That Change the Answer

*A hands-on how-to for the Cosmos3-Edge reasoner on images and video, from your first call to a tuned setup*

**By Obinna Edeh**, Head of Physical AI Lab, EmbodiedEdge

**The companion guide.** [Cosmos Reason 2 for First-Timers](https://colab.research.google.com/github/obiedeh/cosmos-reason2-first-timers-guide/blob/main/cosmos_reason2_first_timers_guide.ipynb) runs the same tests on Cosmos Reason 2 (2B), the model this guide was first written for. If you start here, you can go back to it at any time. The two can be read in either order.

**Why I built this.** I wrote [the same kind of guide for Cosmos Reason 2](https://github.com/obiedeh/cosmos-reason2-first-timers-guide), the model behind my [Urban Edge Vision Analytics](https://github.com/obiedeh/urban-edge-vision-analytics) and [Physical AI Safety Observability](https://github.com/obiedeh/physical-ai-safety-observability) projects. NVIDIA has since released Cosmos 3, and its smallest model, Cosmos3-Edge, is the one sized for the boards I work on. The settings I learned on Reason 2 do not all carry over. Thinking is switched on by default, a video is turned into tokens differently, and 4-bit loading, which I used on Reason 2, returned no readable text here. So I ran the same tests again on Cosmos3-Edge and wrote down what each setting does to accuracy and response time.

**Why Google Colab.** I built it for Colab on purpose, to make the model easier to learn and adopt. It runs in a browser with nothing to install on your own machine, so a first-timer can go from zero to a working call without owning a GPU.

This notebook takes you from an empty Google Colab session to working image and video calls on **NVIDIA Cosmos3-Edge**, then measures how each main parameter changes accuracy, thinking and response time. Every claim about a parameter is backed by a test with a known answer that you run yourself.

**What Cosmos3-Edge is.** Cosmos 3 is NVIDIA's family of world models for physical AI. Each model has two parts. The **reasoner** reads text, images and video and answers in text. The **generator** produces video, images and robot actions. This guide covers the reasoner only: it can caption, answer questions, locate objects with bounding boxes, timestamp events in video, plan a robot's next move, and think step by step before answering. The Hugging Face `transformers` library loads just the reasoner from the checkpoint, about 2.4 billion of the model's 4 billion parameters.

**The three sizes, and why this guide uses Edge.** NVIDIA publishes Cosmos 3 in three sizes: [Cosmos3-Edge](https://huggingface.co/nvidia/Cosmos3-Edge) (4B), [Cosmos3-Nano](https://huggingface.co/nvidia/Cosmos3-Nano) (16B) and [Cosmos3-Super](https://huggingface.co/nvidia/Cosmos3-Super) (64B). Edge is the smallest, so it has the best chance of fitting a Colab GPU, and NVIDIA lists Jetson boards among its targets. The settings you learn here are worth testing on the larger models too, though their scores will differ and the larger ones load through a different class. NVIDIA's own code and guides are at [github.com/NVIDIA/cosmos](https://github.com/NVIDIA/cosmos).

**What you will do**

| Part | Topic |
|---|---|
| 1 | Set up Colab, install, load the model, build the helpers |
| 2 | First image call and first video call |
| 3 | What is inside a call: messages, the prompt, tokens |
| 4 | `max_new_tokens`: output length and time |
| 5 | Vision token budget: image detail, accuracy and time. Cropping. |
| 6 | Video: `fps`, the shared frame budget, long video |
| 7 | Context window: counting what you send |
| 8 | Where the time goes: the first token against the rest |
| 9 | Thinking mode: on by default, accuracy against time |
| 10 | Sampling: `temperature`, `top_p`, `top_k`, `repetition_penalty`, `seed` |
| 11 | Prompt structure: media order and the system prompt, tested |
| 12 | Knowing when it is wrong: leading questions and answer agreement |
| 13 | Real footage check: a warehouse camera |
| 14 | Task gallery: NVIDIA's prompts on NVIDIA's footage |
| 15 | Several images, and follow-up questions |
| 16 | Speed and memory: batching, 16-bit formats and 4-bit loading |
| 17 | Streaming, and the same parameters on a vLLM server |
| 18 | Tune it on your own data |
| 19 | Cheat sheet and troubleshooting |

Parts 1 to 3 are the base. After Part 1 every part runs independently, so you can jump straight to the one you need. A full run makes a few hundred short calls.

**Before you start**

1. In Colab choose **Runtime > Change runtime type > T4 GPU** (or any GPU).
2. That is the only required step. The model is not gated, so there is no license page to accept and the download works without an account. It is released under the [OpenMDW 1.1](https://openmdw.ai/license/1-1/) license.
3. Optional: a Hugging Face read token makes the download faster and avoids rate limits. Create one at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens). In Colab, click the key icon in the left bar, add a secret named `HF_TOKEN` and switch on notebook access.

The reasoner holds about 4.9 GB on the GPU. Every result table prints the peak GPU memory so you can see it on your own runtime. The high point is Part 16, which loads a second copy of the model for a comparison.

NVIDIA states that it tested the model in BF16 precision only. This notebook loads it in float16, the other 16-bit format, because older GPUs such as the T4 do not run BF16 natively. Part 16 compares the two.

## Part 1. Setup

### 1.1 Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found. In Colab: Runtime > Change runtime type > T4 GPU."
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}, {gpu.total_memory / 1e9:.0f} GB   torch {torch.__version__}")

### 1.2 Install

Colab already has PyTorch, OpenCV, Pillow, pandas and matplotlib. The main install is `transformers`. Cosmos3-Edge needs a recent release, and this notebook pins 5.23.0, the version every result here was measured on. `bitsandbytes` is only needed for the 4-bit experiment in Part 16.

In [ ]:
%pip install -q "transformers==5.23.0" accelerate bitsandbytes

### 1.3 Model name, and an optional login

The model downloads without a login. If you added an `HF_TOKEN` secret, this cell uses it, which makes the download faster.

In [ ]:
import os
MODEL_ID = os.environ.get("COSMOS_MODEL", "nvidia/Cosmos3-Edge")   # set COSMOS_MODEL to a local folder to skip the download

if not os.path.isdir(MODEL_ID):
    try:
        from google.colab import userdata
        from huggingface_hub import login
        login(token=userdata.get("HF_TOKEN"))
        print("Logged in with the HF_TOKEN secret.")
    except Exception:
        print("No HF_TOKEN secret found. Downloading without a login, which works because this model is not gated.")

### 1.4 Load the model and the processor

Two objects do all the work:

- the **processor** turns your text, images and video frames into numbers (tokens and pixel tensors),
- the **model** reads those numbers and generates the answer, one token at a time.

The load arguments:

| Argument | Meaning |
|---|---|
| `AutoModelForImageTextToText` | Picks the right class for this checkpoint (`Cosmos3EdgeForConditionalGeneration`). It loads the reasoner only. |
| `dtype=torch.float16` | Store weights as 16-bit floats. Half the memory of 32-bit. NVIDIA's own sample uses `torch.bfloat16`; Part 16 compares the two. |
| `device_map="auto"` | Put the model on the GPU. |

The first run downloads about 7.7 GB. That file set also holds weights the reasoner does not use, so less than that ends up on the GPU.

In [ ]:
import time, warnings
import transformers
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

t0 = time.perf_counter()
model = transformers.AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="auto"
)
processor = transformers.AutoProcessor.from_pretrained(MODEL_ID)
model.eval()
print(f"Loaded in {time.perf_counter() - t0:.0f} s")
print(f"GPU memory held by the weights: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print(f"Parameters loaded (the reasoner): {model.num_parameters() / 1e9:.2f} billion")

from importlib.metadata import version
print("Library versions:", {name: version(name) for name in ("transformers", "accelerate", "bitsandbytes")})

### 1.5 Get the sample media

The same media as my Cosmos Reason 2 guide, so the two models can be compared on identical footage. Two files from NVIDIA's `cosmos-reason2` repository (a still of a two-handed robot at a table, and a short dashcam clip), plus real footage from NVIDIA's Cosmos Cookbook prompt guide: a packing robot, a drive into a construction zone, a generated block-stacking clip, a warehouse camera, a doorstep camera and a forklift photo. About 50 MB in total, fetched from NVIDIA's sites at run time.

In [ ]:
import urllib.request
import cv2, numpy as np
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Video

REPO = "https://raw.githubusercontent.com/nvidia-cosmos/cosmos-reason2/main/assets/"
GUIDE = "https://nvidia-cosmos.github.io/cosmos-cookbook/getting_started/prompt_guide/assets/"

def fetch(name, base):
    if not os.path.exists(name):
        try:
            urllib.request.urlretrieve(base + name, name)
        except Exception as e:
            print(f"Could not download {name}: {e}. Parts that use it will fail; the others still run.")
    return name

ROBOT_IMAGE     = fetch("sample.png", REPO)
DASHCAM_VIDEO   = fetch("sample.mp4", REPO)          # 2.5 s dashcam clip
PACKING_VIDEO   = fetch("agibot.mp4", GUIDE)         # 6.5 s, two robot arms packing a box
DRIVING_VIDEO   = fetch("ego_v.mp4", GUIDE)          # 15 s, driving into a construction zone
GENERATED_VIDEO = fetch("sdg.mp4", GUIDE)            # 6 s, AI-generated block stacking
WAREHOUSE_VIDEO = fetch("warehouse_1.mp4", GUIDE)    # 46 s, warehouse aisle camera
DOORSTEP_VIDEO  = fetch("package.mp4", GUIDE)        # 12 s, doorstep camera
FORKLIFT_IMAGE  = fetch("load.png", GUIDE)

def frame_strip(path, n=4, height=150):
    # n evenly spaced frames of a video, side by side, so you can see what the model is given
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    tiles = []
    for i in np.linspace(0, total - 1, n).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame = cap.read()
        if ok:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            tiles.append(cv2.resize(frame, (int(frame.shape[1] * height / frame.shape[0]), height)))
    cap.release()
    return Image.fromarray(np.hstack(tiles))

img = Image.open(ROBOT_IMAGE)
print("Robot image size:", img.size)
display(img.convert("RGB").resize((640, 400)))

### 1.6 Build test media with known answers

Real footage shows what the model is for. To *measure* a parameter you also need media where the right answer is certain. This cell draws three test files:

- **`labels.png`**: six boxes, each with a small printed code. Reading the codes tests fine detail.
- **`digits.mp4`**: six digits flash on screen for half a second each. Listing them tests whether short events are caught.
- **`long_digits.mp4`**: the same idea over one minute with twenty digits. It tests long video.

In [ ]:
CODES = ["KX-4827", "BT-9153", "QM-3068", "ZR-7742", "HD-5291", "LV-1836"]

def make_label_image(path="labels.png", codes=CODES):
    im = Image.new("RGB", (1600, 1000), (205, 200, 190))
    d = ImageDraw.Draw(im)
    colors = [(170, 120, 70), (90, 120, 160), (120, 150, 100), (160, 100, 100), (130, 110, 150), (180, 160, 90)]
    small, big = ImageFont.load_default(size=22), ImageFont.load_default(size=40)
    for i, code_text in enumerate(codes):
        x, y = 120 + (i % 3) * 480, 140 + (i // 3) * 440
        d.rectangle([x, y, x + 360, y + 300], fill=colors[i], outline=(40, 40, 40), width=4)
        d.rectangle([x + 110, y + 120, x + 250, y + 170], fill="white", outline="black", width=2)
        d.text((x + 180, y + 145), code_text, font=small, fill="black", anchor="mm")
        d.text((x + 180, y + 50), f"BOX {i + 1}", font=big, fill="white", anchor="mm")
    im.save(path)
    return path

DIGITS = [7, 3, 9, 4, 8, 2]
LONG_DIGITS = [7, 3, 9, 4, 8, 2, 5, 1, 6, 3, 8, 4, 2, 9, 7, 5, 1, 6, 4, 3]

def make_digit_video(path, digits, gap, native_fps=24, visible_for=0.5):
    # Each digit is on screen for `visible_for` seconds; a new one starts every `gap` seconds.
    seconds = 1.0 + gap * len(digits)
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), native_fps, (640, 360))
    starts = [1.0 + i * gap for i in range(len(digits))]
    for i in range(int(seconds * native_fps)):
        t = i / native_fps
        frame = np.full((360, 640, 3), 40, np.uint8)
        cv2.circle(frame, (int(40 + 560 * t / seconds), 300), 20, (255, 160, 0), -1)      # a ball that crosses the screen
        for start, digit in zip(starts, digits):
            if start <= t < start + visible_for:
                cv2.putText(frame, str(digit), (270, 220), cv2.FONT_HERSHEY_SIMPLEX, 6, (255, 255, 255), 14)
        writer.write(frame)
    writer.release()
    return path, seconds

LABEL_IMAGE = make_label_image()
DIGIT_VIDEO, DIGIT_SECONDS = make_digit_video("digits.mp4", DIGITS, gap=1.8)
LONG_VIDEO, LONG_SECONDS = make_digit_video("long_digits.mp4", LONG_DIGITS, gap=3.0)

print(f"labels.png: codes {CODES}")
display(Image.open(LABEL_IMAGE).resize((640, 400)))
print(f"digits.mp4: {DIGIT_SECONDS:.0f} s, digits {DIGITS}.   long_digits.mp4: {LONG_SECONDS:.0f} s, {len(LONG_DIGITS)} digits.")
display(frame_strip(DIGIT_VIDEO, n=6, height=110))

### 1.7 One helper for every call: `ask()`

Every call in this notebook goes through the function below, so each parameter you will study is one named argument. It does the four steps of any call:

1. **Build the conversation**: a system message, then a user message with the media first and the text after it. NVIDIA's prompt guide for the Cosmos 3 reasoner gives both rules: media before text, and the system prompt `You are a helpful assistant.`
2. **Prepare inputs**: the processor resizes the image or frames to fit the vision token budget and converts everything to tensors.
3. **Generate**: the model produces up to `max_new_tokens` new tokens.
4. **Decode and measure**: turn tokens back into text and record token counts, seconds and peak GPU memory.

One thing is different from most models, and it matters from the first call. **Cosmos3-Edge thinks before it answers unless you tell it not to.** The switch is `enable_thinking` in the chat template, and it is on by default. `ask()` turns it off, so a plain call returns a plain answer, and turns it on only when you pass `reasoning=True`. Part 9 measures what thinking costs and what it buys.

Video frames are read here with OpenCV instead of the library's built-in loader. The built-in loader depends on video packages that differ between Colab images, and reading frames yourself also shows exactly what `fps` does: it sets how many frames per second of video the model gets to see.

In [ ]:
import functools, re, json
import pandas as pd
import matplotlib.pyplot as plt
from transformers.video_utils import VideoMetadata
from transformers.generation.streamers import BaseStreamer
pd.set_option("display.max_colwidth", 90)

PIXELS_PER_TOKEN = 32 * 32      # one vision token covers a 32 x 32 pixel patch
SYSTEM_PROMPT = "You are a helpful assistant."

# The answer-format instruction from NVIDIA's Cosmos 3 reasoner prompt guide. Used only by reasoning="prompt" (Part 9).
REASONING_PROMPT = '''Answer the question using the following format:

<think>
Your reasoning.
</think>

Write your final answer immediately after the </think> tag.'''


def final_answer(text, thinking=False, hit_limit=False):
    # With thinking on, the opening <think> is already in the prompt, so the output is "reasoning </think> answer".
    #   1. </think> is there: the answer is what follows it.
    #   2. No </think> and the token limit was reached: the reasoning never finished, so there is no answer.
    #   3. No </think> and the model stopped by itself: it answered straight away, so everything it wrote is the answer.
    if "</think>" in text:
        return text.split("</think>")[-1].strip()
    return "" if (thinking and hit_limit) else text.strip()


@functools.lru_cache(maxsize=6)
def read_video(path, fps, clip=None):
    # Pick `fps` evenly spaced frames per second of video, from the whole file or from clip=(start_s, end_s).
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    native_fps = cap.get(cv2.CAP_PROP_FPS)
    first, last = (0, total - 1) if clip is None else (int(clip[0] * native_fps), min(int(clip[1] * native_fps), total) - 1)
    n = min(max(int((last - first + 1) / native_fps * fps), 4), 768, last - first + 1)   # same limits as the model's video processor: 4 to 768 frames
    wanted = np.linspace(first, last, n).round().astype(int)
    keep, frames = set(wanted.tolist()), []
    for i in range(last + 1):
        if not cap.grab():
            break
        if i in keep:
            frames.append(cv2.cvtColor(cap.retrieve()[1], cv2.COLOR_BGR2RGB))
    cap.release()
    # frames_indices keeps each frame's position in the original file, so timestamps stay true even for a clip
    meta = VideoMetadata(total_num_frames=total, fps=native_fps, frames_indices=wanted[:len(frames)].tolist(),
                         duration=total / native_fps, video_backend="opencv")
    return np.stack(frames), meta


def build_inputs(prompt, image=None, video=None, clip=None, fps=4, max_vision_tokens=1024, min_vision_tokens=16,
                 reasoning=False, system=SYSTEM_PROMPT, media_first=True):
    # Steps 1 and 2: conversation -> prompt text -> tensors. `image` can be one image or a list of images.
    images = [] if image is None else (list(image) if isinstance(image, (list, tuple)) else [image])
    media = [{"type": "image"} for _ in images] + ([{"type": "video"}] if video is not None else [])
    # reasoning: False = thinking switched off, True = thinking on, "prompt" = thinking on plus NVIDIA's answer-format instruction
    text = {"type": "text", "text": prompt + ("\n\n" + REASONING_PROMPT if reasoning == "prompt" else "")}
    conversation = [{"role": "system", "content": [{"type": "text", "text": system}]}] if system else []
    conversation.append({"role": "user", "content": media + [text] if media_first else [text] + media})
    prompt_text = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True,
                                                enable_thinking=bool(reasoning))

    extra, n_frames = {}, None
    if images:
        extra["images"] = [(Image.open(i) if isinstance(i, str) else i).convert("RGB") for i in images]
    if video is not None:
        frames, meta = read_video(video, fps, clip)
        n_frames = len(frames)
        extra.update(videos=[frames], video_metadata=[meta], do_sample_frames=False)
    size = {"shortest_edge": min_vision_tokens * PIXELS_PER_TOKEN, "longest_edge": max_vision_tokens * PIXELS_PER_TOKEN}
    return processor(text=[prompt_text], return_tensors="pt", size=size, **extra), n_frames


def count_tokens(inputs):
    ids = inputs["input_ids"][0]
    vision = int(((ids == model.config.image_token_id) | (ids == model.config.video_token_id)).sum())
    return len(ids), vision


class FirstTokenTimer(BaseStreamer):
    # Records the moment the first new token arrives.
    def __init__(self):
        self.calls, self.at = 0, None
    def put(self, value):
        self.calls += 1
        if self.calls == 2:             # call 1 is the prompt itself, call 2 is the first generated token
            torch.cuda.synchronize()
            self.at = time.perf_counter()
    def end(self):
        pass


def ask(prompt, image=None, video=None, clip=None, fps=4, max_vision_tokens=1024, min_vision_tokens=16,
        reasoning=False, system=SYSTEM_PROMPT, media_first=True, max_new_tokens=256, do_sample=False, seed=0,
        streamer=None, use_model=None, **generate_kwargs):
    m = use_model or model
    t0 = time.perf_counter()
    inputs, n_frames = build_inputs(prompt, image, video, clip, fps, max_vision_tokens, min_vision_tokens, reasoning, system, media_first)
    inputs = inputs.to(m.device)
    t1 = time.perf_counter()
    n_in, n_vision = count_tokens(inputs)

    timer = FirstTokenTimer() if streamer is None else None
    torch.cuda.reset_peak_memory_stats()
    transformers.set_seed(seed)
    with torch.inference_mode():
        out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=do_sample, streamer=streamer or timer, **generate_kwargs)
    torch.cuda.synchronize()
    t2 = time.perf_counter()

    new_tokens = out[0][n_in:]
    text = processor.decode(new_tokens, skip_special_tokens=True).strip()
    return {
        "text": text,
        "answer": final_answer(text, thinking=bool(reasoning), hit_limit=len(new_tokens) >= max_new_tokens),
        "input_tokens": n_in,
        "vision_tokens": n_vision,
        "output_tokens": len(new_tokens),
        "frames": n_frames,
        "prep_s": round(t1 - t0, 2),
        "gen_s": round(t2 - t1, 2),
        "first_token_s": round(timer.at - t1, 2) if timer and timer.at else None,
        "tokens_per_s": round(len(new_tokens) / (t2 - t1), 1),
        "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
        "hit_limit": len(new_tokens) >= max_new_tokens,     # True means the answer was cut off
    }


STAT_KEYS = ["input_tokens", "vision_tokens", "output_tokens", "frames", "prep_s", "gen_s", "first_token_s", "tokens_per_s", "peak_gpu_gb", "hit_limit"]

def show(result):
    print(result["text"])
    print("\n" + "  ".join(f"{k}={result[k]}" for k in STAT_KEYS if result[k] is not None))

def parse_json(text):
    match = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    return json.loads(match.group(1) if match else text)

_ = ask("Hi", image=ROBOT_IMAGE, max_new_tokens=4)      # warm-up: the first call on a fresh GPU is slow
print("Ready.")

### 1.8 One helper for every measurement: `evaluate()`

To say a setting is "more accurate" you need questions with known answers and a way to score them. `evaluate()` takes a list of questions, runs each through `ask()` with whatever settings you pass, and returns the score and the time. `sweep()` repeats that over several values of one setting. Part 18 reuses both on your own data.

A question is a dictionary: the media, the question text `q`, and the `truth`. The truth can be `"yes"` or `"no"`, a number, a word that must appear, or a function that returns `True` for a right answer. A number is scored strictly: the answer must name that number and no other, so a long answer that lists several numbers counts as wrong even if the right one is among them.

In [ ]:
NUMBER_WORDS = {"zero": "0", "one": "1", "two": "2", "three": "3", "four": "4", "five": "5", "six": "6",
                "seven": "7", "eight": "8", "nine": "9"}

def is_correct(answer, truth):
    if callable(truth):
        return bool(truth(answer))
    a, t = answer.lower().strip(), str(truth).lower()
    if t in ("yes", "no"):
        return a.startswith(t)
    if t.isdigit():
        # The answer must name this number and no other, as digits or as a number word.
        numbers = re.findall(r"\d+", a) or [digit for word, digit in NUMBER_WORDS.items() if word in re.findall(r"[a-z]+", a)]
        return bool(numbers) and set(numbers) == {t}
    return t in a

def evaluate(questions, **settings):
    # Ask every question with the same settings. Returns a one-row summary and the per-question table.
    settings.setdefault("max_new_tokens", 800 if settings.get("reasoning") else 40)
    rows = []
    for item in questions:
        media = {k: v for k, v in item.items() if k not in ("q", "truth")}
        r = ask(item["q"], **{**media, **settings})
        answer = r["answer"]
        rows.append({"question": item["q"][:62], "truth": "(function)" if callable(item["truth"]) else item["truth"],
                     "answer": answer[:62] if answer else "(no answer: cut off while thinking)",
                     "correct": is_correct(answer, item["truth"]), "output_tokens": r["output_tokens"],
                     "gen_s": r["gen_s"], "peak_gpu_gb": r["peak_gpu_gb"]})
    df = pd.DataFrame(rows)
    summary = {"correct": int(df["correct"].sum()), "questions": len(df), "accuracy": round(float(df["correct"].mean()), 2),
               "total_gen_s": round(float(df["gen_s"].sum()), 2), "avg_output_tokens": round(float(df["output_tokens"].mean()), 1),
               "peak_gpu_gb": float(df["peak_gpu_gb"].max())}
    return summary, df

# Shared test prompts and scorers, used by several parts
READ_CODES = ("Read the code printed on the white label of each box, from BOX 1 to BOX 6. "
              "Answer with the six codes separated by commas, nothing else.")
LIST_DIGITS = ("Large white digits flash on the screen one at a time during this video. "
               "List every digit that appears, in order, separated by commas. Answer with the digits only.")

def codes_read(text):
    return sum(code_text in text for code_text in CODES)      # how many of the six codes were read correctly

def digits_said(text):
    return [int(x) for x in re.findall(r"\d", text)]      # digits only: an answer spelled out in words does not count

# Six questions about the warehouse clip, with answers checked by eye against the footage (Parts 11 and 13)
WAREHOUSE_QUESTIONS = [
    dict(q="How many different people appear in this video? Answer with a number.", truth="2"),
    dict(q="Does every person in this video wear a hard hat? Answer yes or no.", truth="no"),
    dict(q="Is a cardboard box left lying on the floor at some point in this video? Answer yes or no.", truth="yes"),
    dict(q="One person picks a cardboard box up off the floor. Is that person wearing a high-visibility vest? Answer yes or no.", truth="no"),
    dict(q="There are signs with single letters on the shelf posts. Which letters are shown? Answer with the letters only.",
         truth=lambda a: sorted(set(re.findall(r"\b([a-z])\b", a.lower()))) == list("cdef")),
    dict(q="What color is the hard hat worn by the first worker? One word.", truth="yellow"),
]

def sweep(questions, name, values, **fixed):
    # Run evaluate() once per value of one setting and stack the summaries.
    return pd.DataFrame([{name: v, **evaluate(questions, **{**fixed, name: v})[0]} for v in values])

def two_panel(df, x, title):
    # Accuracy and time against one setting.
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.2))
    time_col = "gen_s" if "gen_s" in df else "total_gen_s"
    a.plot(df[x], df["accuracy"] * 100, marker="o", color="#2a7f62")
    a.set_ylim(-5, 105); a.set_ylabel("accuracy (%)"); a.set_xlabel(x); a.set_title("Accuracy")
    b.plot(df[x], df[time_col], marker="o", color="#b5532a")
    b.set_ylim(bottom=0); b.set_ylabel("generation time (s)"); b.set_xlabel(x); b.set_title("Response time")
    for ax in (a, b):
        ax.set_xscale("log", base=2); ax.set_xticks(list(df[x])); ax.set_xticklabels([str(v) for v in df[x]]); ax.grid(alpha=0.3)
    fig.suptitle(title); fig.tight_layout(); plt.show()

## Part 2. First calls

### 2.1 An image question

In [ ]:
r = ask("Describe this image in two sentences.", image=ROBOT_IMAGE)
show(r)

Read the stats line under the answer. You will use it in every part:

- `input_tokens`: everything the model read. `vision_tokens` is the share that came from the picture. Text is a small fraction.
- `output_tokens`: what the model wrote.
- `prep_s`: seconds to load and resize the media. `gen_s`: seconds the model spent reading the input and writing the answer. `first_token_s` is the part of `gen_s` that passed before the first word appeared (Part 8).
- `peak_gpu_gb`: highest GPU memory during the call.
- `hit_limit`: `True` if the answer stopped because it reached `max_new_tokens`.

### 2.2 A video question

In [ ]:
display(Video(DASHCAM_VIDEO, embed=True, width=480))
r = ask("Caption the video in detail.", video=DASHCAM_VIDEO, fps=4, max_new_tokens=400)
show(r)

A caption reads fluently whether or not it is true. Compare it with the clip above before you trust it. Part 12 shows how to test that systematically.

## Part 3. What is inside a call

The model never sees your Python objects. The processor writes one long prompt in the model's chat format, with a block of placeholder tokens where the picture goes. Each placeholder is later replaced by one patch of the image. Printing the prompt (with the placeholders collapsed) makes the structure visible.

Look at the last line of the printed prompt. It ends with `<think></think>`: an empty thinking block, already closed. That is what "thinking off" looks like to the model. With thinking on, the prompt ends with an open `<think>` and the model fills it in.

In [ ]:
def show_prompt(inputs):
    text = processor.decode(inputs["input_ids"][0])
    text = re.sub(r"(<\|(image|video)_pad\|>)+", lambda m: f"[{m.group(0).count('_pad')} vision tokens]", text)
    print(text)

inputs, _ = build_inputs("What objects are on the table?", image=ROBOT_IMAGE, max_vision_tokens=1024)
show_prompt(inputs)
print("\ntotal, vision tokens:", count_tokens(inputs))

For video, the processor writes a timestamp in front of every frame, and every frame gets its own block of vision tokens. Those timestamps are how the model can tell you *when* something happened.

In [ ]:
inputs, n_frames = build_inputs("Caption the video in detail.", video=DASHCAM_VIDEO, fps=4)
print("frames sent:", n_frames)
show_prompt(inputs)

Three facts to carry forward:

1. **Cost is counted in tokens.** Time and memory grow with `input_tokens + output_tokens`.
2. **Vision dominates the input.** One vision token covers a 32 x 32 pixel patch, so a 1024 x 1024 image is about 1,024 tokens. In video, each frame is tokenized separately, so the cost grows with every frame you send.
3. **You control both sides.** The vision budget and `fps` set the input size. `max_new_tokens` caps the output size.

## Part 4. `max_new_tokens`: output length and time

`max_new_tokens` is a hard cap on how many tokens the model may write. It does not make the model write more. The model stops by itself when it finishes. The cap only matters when it is lower than the answer the model wanted to give, and then the answer is cut mid-sentence.

In [ ]:
rows = []
for cap in (16, 64, 256, 1024):
    r = ask("Describe this image in detail.", image=ROBOT_IMAGE, max_new_tokens=cap)
    rows.append({"max_new_tokens": cap, "output_tokens": r["output_tokens"], "gen_s": r["gen_s"],
                 "hit_limit": r["hit_limit"], "answer ends with": "..." + r["text"][-55:]})
pd.DataFrame(rows)

What to read in the table:

- `gen_s` rises with `output_tokens`, because the model writes one token per step.
- Rows with `hit_limit = True` end mid-sentence.
- Once the cap is above the natural answer length, raising it changes nothing: the last two rows are the same answer in the same time.

**Rule:** set `max_new_tokens` above the longest answer you expect, and check `hit_limit`. NVIDIA recommends 4096 or more when thinking is on (Part 9), because a cut-off thinking trace never reaches the answer.

## Part 5. Vision token budget: image detail, accuracy and time

### 5.1 The budget

Before the model sees an image, the processor resizes it so the number of vision tokens falls between a minimum and a maximum. In `ask()` these are `min_vision_tokens` and `max_vision_tokens`. Under the hood they set the processor's `size` (`shortest_edge` and `longest_edge`, measured in pixels: tokens x 1024). NVIDIA's server command calls the same thing `min_pixels` and `max_pixels` (Part 17).

A smaller budget means a smaller picture: faster, less memory, and fine detail is lost. The test is the label image from Part 1: read six small printed codes.

In [ ]:
print("Prompt:", READ_CODES, "\n")
rows = []
for budget in (64, 128, 256, 512, 1024, 2048):
    r = ask(READ_CODES, image=LABEL_IMAGE, max_vision_tokens=budget, max_new_tokens=80)
    rows.append({"max_vision_tokens": budget, "vision_tokens": r["vision_tokens"], "codes_correct": f"{codes_read(r['text'])}/6",
                 "accuracy": codes_read(r["text"]) / 6, "gen_s": r["gen_s"], "peak_gpu_gb": r["peak_gpu_gb"], "answer": r["text"]})
vision_df = pd.DataFrame(rows)
display(vision_df)
two_panel(vision_df, "max_vision_tokens", "Reading small text: vision token budget")

What to read here:

- At the smallest budgets the image is shrunk until the printed codes blur, and the model guesses characters. It does not say "I cannot read this". It returns confident wrong codes. **A budget that is too low fails silently.**
- Past the point where the text is legible, more tokens add time and memory and no accuracy.
- `vision_tokens` stops growing once the budget exceeds the image's own size, because the processor does not enlarge an image to fill the budget.

**Rule:** pick the smallest budget at which your smallest important detail is still readable. Scene-level questions ("is there a person?") need few tokens. Small text, gauges and distant objects need many.

### 5.2 Cropping: spend the tokens where the detail is

The budget is spread over the whole picture. If you know where the detail is, cut that region out and send it alone: the same number of tokens now covers a smaller area, so each token sees finer detail. The cell compares the whole label image at 96 tokens, the same image as two halves at 96 tokens each, and the whole image at 192 tokens.

In [ ]:
READ_ANY = "Read the code printed on the white label of each box. Answer with the codes separated by commas, nothing else."
label = Image.open(LABEL_IMAGE)
top, bottom = label.crop((0, 0, 1600, 500)), label.crop((0, 500, 1600, 1000))

whole_small = ask(READ_ANY, image=label, max_vision_tokens=96, max_new_tokens=80)
halves = [ask(READ_ANY, image=part, max_vision_tokens=96, max_new_tokens=80) for part in (top, bottom)]
whole_double = ask(READ_ANY, image=label, max_vision_tokens=192, max_new_tokens=80)

pd.DataFrame([
    {"approach": "whole image, 96 tokens", "calls": 1, "vision_tokens": whole_small["vision_tokens"],
     "codes_correct": f"{codes_read(whole_small['text'])}/6", "gen_s": whole_small["gen_s"]},
    {"approach": "two halves, 96 tokens each", "calls": 2, "vision_tokens": sum(h["vision_tokens"] for h in halves),
     "codes_correct": f"{sum(codes_read(h['text']) for h in halves)}/6", "gen_s": round(sum(h["gen_s"] for h in halves), 2)},
    {"approach": "whole image, 192 tokens", "calls": 1, "vision_tokens": whole_double["vision_tokens"],
     "codes_correct": f"{codes_read(whole_double['text'])}/6", "gen_s": whole_double["gen_s"]},
])

Compare the first two rows: the picture and the per-call budget are the same, and only the framing changed. Then compare the last two rows, which spend about the same total tokens. Cropping costs an extra call, and in return each call sees the detail at twice the scale.

**Rule:** when a detail is too small, you can raise the budget or crop. Cropping is the one to reach for when the budget is already as high as your time or memory allows, or when the detail sits in a small part of a large frame. A detector or a fixed region of interest (a gauge, a label, a doorway) tells you where to crop.

## Part 6. Video: `fps`, the shared frame budget, long video

### 6.1 `fps`: how often the model looks

`fps` is the number of frames per second of video handed to the model. It is separate from the frame rate the file was recorded at. NVIDIA recommends 4 frames per second for video sent to the Cosmos 3 reasoner.

Lower `fps` means fewer frames and a faster call. The risk is that anything shorter than the gap between sampled frames can be missed. The digit clip from Part 1 is built to show that: each digit is on screen for half a second.

In [ ]:
frames, _ = read_video(DIGIT_VIDEO, fps=1)
display(Image.fromarray(np.hstack([frames[i][::2, ::2] for i in range(0, len(frames), 2)])))
print("Above: every second frame the model receives at fps=1. Count how many digits made it in.")

print("Prompt:", LIST_DIGITS, "\n")
rows = []
for fps in (0.5, 1, 2, 4, 8):
    r = ask(LIST_DIGITS, video=DIGIT_VIDEO, fps=fps, max_vision_tokens=2048, max_new_tokens=40)
    rows.append({"fps": fps, "frames": r["frames"], "vision_tokens": r["vision_tokens"],
                 "accuracy": float(digits_said(r["text"]) == DIGITS),      # all six, in order, nothing extra
                 "prep_s": r["prep_s"], "gen_s": r["gen_s"], "peak_gpu_gb": r["peak_gpu_gb"], "answer": r["text"]})
fps_df = pd.DataFrame(rows)
display(fps_df)
two_panel(fps_df, "fps", "Catching short events: frames per second")

What to read here:

- At low `fps` the sampled frames step over most of the digits. The model reports the few it saw. Again, it does not warn you.
- Once the gap between frames is shorter than the event (0.5 s here, so 2 fps and up), every digit lands in at least one frame and the answer is right.
- `frames` doubles with each row, but `vision_tokens` does not. That is the shared budget, next.

**Rule:** choose `fps` from the shortest event you must catch. You need at least one sampled frame inside it, and two is safer. Use 4 when unsure, since that is NVIDIA's recommended value.

### 6.2 The frame budget is shared

For video, `max_vision_tokens` is one budget for the **whole clip**. More frames means fewer pixels per frame. So with a fixed budget, raising `fps` trades sharpness for coverage in time. The next cell fixes `fps=4` and changes only the budget.

In [ ]:
rows = []
for budget in (256, 512, 1024, 2048, 4096):
    r = ask(LIST_DIGITS, video=DIGIT_VIDEO, fps=4, max_vision_tokens=budget, max_new_tokens=40)
    rows.append({"max_vision_tokens": budget, "frames": r["frames"], "vision_tokens": r["vision_tokens"],
                 "tokens_per_frame": round(r["vision_tokens"] / r["frames"], 1),
                 "accuracy": float(digits_said(r["text"]) == DIGITS), "gen_s": r["gen_s"], "answer": r["text"]})
pd.DataFrame(rows)

`frames` is the same in every row, and `tokens_per_frame` shows how much picture each moment of the video gets. With only a few tokens per frame the digits are unreadable and the model makes a sequence up. The clip has enough frames and still fails, because each frame is too coarse. Watch for a row where the model answers in words ("seven, three") after being asked for digits: that is a sign it is close to the limit of what it can see.

**Rule:** for video, think in tokens per frame. Work out `frames = seconds x fps`, decide how many tokens one frame needs for your detail, and multiply. The default of 1024 in `ask()` suits an image. A 12 second clip at 4 fps needed about twice that here.

### 6.3 Long video: one call or several clips?

A one minute video at 4 fps is 240 frames, and a lot happens in it. You have two ways to send it. One call, where the budget is spread over every frame and the model has to keep track of every event. Or several clips, one call each: `ask()` takes `clip=(start_seconds, end_seconds)` for this, and the timestamps the model sees stay true to the original file.

The test: list all twenty digits in the one minute clip. Twice in a single call, at two frame rates, then split three ways.

In [ ]:
def list_digits_in_clips(n_clips, budget):
    found, tokens, frames, seconds = [], 0, 0, 0.0
    for k in range(n_clips):
        clip = (k * LONG_SECONDS / n_clips, (k + 1) * LONG_SECONDS / n_clips)
        r = ask(LIST_DIGITS, video=LONG_VIDEO, clip=clip, fps=4, max_vision_tokens=budget, max_new_tokens=40)
        found += digits_said(r["text"]); tokens += r["vision_tokens"]; frames += r["frames"]; seconds += r["gen_s"]
    return found, tokens, frames, seconds

runs = []
for fps, budget in ((4, 4096), (2, 8192)):
    r = ask(LIST_DIGITS, video=LONG_VIDEO, fps=fps, max_vision_tokens=budget, max_new_tokens=120)
    runs.append((f"one call, {fps} fps, {budget} tokens", digits_said(r["text"]), r["vision_tokens"], r["frames"], r["gen_s"]))
for n_clips, budget in ((8, 1024), (12, 1024), (8, 128)):
    runs.append((f"{n_clips} clips of {LONG_SECONDS / n_clips:.1f} s, 4 fps, {budget} tokens each", *list_digits_in_clips(n_clips, budget)))

print("truth:", LONG_DIGITS, f"({len(LONG_DIGITS)} digits)")
pd.DataFrame([{"approach": name, "vision_tokens_total": tokens, "tokens_per_frame": round(tokens / frames), "gen_s_total": round(seconds, 2),
               "exact": found == LONG_DIGITS, "digits_returned": len(found), "answer": str(found)[:95]}
              for name, found, tokens, frames, seconds in runs])

What to read here, using `tokens_per_frame` and `digits_returned` against the true count of twenty:

- **One call at 4 fps** has too few tokens for each of its 244 frames. The model cannot read them and falls back on counting upward.
- **One call at 2 fps with twice the budget** sends half the frames, so each frame gets four times the tokens. Every digit is still on screen for at least one sampled frame. Compare its answer with the truth, digit by digit.
- **Eight clips** gives each call only two or three digits to report. Look at where the clips meet: a digit that straddles a boundary is reported by both neighbors, and a clip that ends on a blank screen can tempt the model to keep counting.
- **Twelve clips** has more boundaries, and the digits near them are lost or doubled.
- **Eight clips at 128 tokens** starves each clip of detail, and the model invents sequences, exactly as in 6.2.

On this model, in my runs, no row returned all twenty digits in order. The single call with enough tokens per frame came closest, and splitting added errors at the boundaries. That is the opposite of what the same test showed on Cosmos Reason 2 (2B), where splitting into eight clips was the only approach that got all twenty.

**Rule:** give each frame enough tokens first. Lower `fps` as far as your shortest event allows, because that frees tokens for every remaining frame. Split into clips only when one call cannot hold the video, and then decide what happens at the boundaries: overlap the clips and remove duplicates, or cut where nothing is happening. Test on footage with a known answer, because the best choice differs from model to model.

For a long video you have three levers, and each one costs something:

| Lever | Gain | Cost |
|---|---|---|
| Lower `fps` | fewer frames, so more tokens for each | short events missed |
| Lower `max_vision_tokens` | fewer tokens | blurrier frames |
| Split into clips, one call per clip | each clip keeps full detail | more calls, errors at the boundaries, and the model cannot connect events across clips |

## Part 7. Context window: counting what you send

The **context window** is the most tokens the model can hold in one call: everything you send plus everything it writes.

`input_tokens + max_new_tokens <= context window`

In [ ]:
print("Model limit (max_position_embeddings):", f"{model.config.text_config.max_position_embeddings:,} tokens")

The model's configuration sets the limit printed above. In practice you hit two earlier limits:

- **GPU memory and time.** Both grow with the token count. A small GPU runs out of memory long before the limit.
- **The server's own limit.** A vLLM server is started with `--max-model-len`. NVIDIA's command for this model uses 131072, and many deployments set it lower. A request whose input is longer than that number is rejected with an error (Part 17).

So before sending a long video, count. `build_inputs()` prepares a call without running the model, which makes counting cheap. The cell below counts tokens for several settings and checks them against a 16,384 token window, a limit you might set on a small server, with 1,024 tokens reserved for the answer. The last row is the one minute clip at a generous budget, which does not fit.

In [ ]:
WINDOW, RESERVED_FOR_ANSWER = 16384, 1024

rows = []
for label_text, kwargs in [
    ("robot image, budget 256",   dict(image=ROBOT_IMAGE, max_vision_tokens=256)),
    ("robot image, budget 1024",  dict(image=ROBOT_IMAGE, max_vision_tokens=1024)),
    ("robot image, budget 4096",  dict(image=ROBOT_IMAGE, max_vision_tokens=4096)),
    ("12 s video, 2 fps, budget 2048", dict(video=DIGIT_VIDEO, fps=2, max_vision_tokens=2048)),
    ("12 s video, 8 fps, budget 2048", dict(video=DIGIT_VIDEO, fps=8, max_vision_tokens=2048)),
    ("12 s video, 8 fps, budget 16384", dict(video=DIGIT_VIDEO, fps=8, max_vision_tokens=16384)),
    ("61 s video, 8 fps, budget 65536", dict(video=LONG_VIDEO, fps=8, max_vision_tokens=65536)),
]:
    inputs, n_frames = build_inputs("Describe what you see.", **kwargs)
    total, vision = count_tokens(inputs)
    rows.append({"call": label_text, "frames": n_frames or "-", "input_tokens": total, "vision_tokens": vision,
                 "fits_16k_window": total + RESERVED_FOR_ANSWER <= WINDOW})
pd.DataFrame(rows)

Now the cost side. The same image question at growing input sizes, with the answer length held short, isolates what input tokens cost in time and memory.

In [ ]:
rows = []
for budget in (256, 1024, 2048, 4096):
    r = ask("What color is the tape? One word.", image=ROBOT_IMAGE, max_vision_tokens=budget, max_new_tokens=8)
    rows.append({"max_vision_tokens": budget, "input_tokens": r["input_tokens"], "gen_s": r["gen_s"],
                 "peak_gpu_gb": r["peak_gpu_gb"], "answer": r["text"]})
pd.DataFrame(rows)

The answer is the same in every row. Only the time and memory changed. Input tokens you do not need are pure cost.

**Rule:** budget the window before the call. Reserve room for the answer first (more if thinking is on), then spend what is left on vision.

## Part 8. Where the time goes: the first token against the rest

A call has two phases, and different settings drive each one:

1. **Before the first token.** The model reads the whole input: all vision tokens and the prompt. This time depends on `input_tokens`, so on the vision budget and `fps`.
2. **After the first token.** The model writes the answer one token at a time. This time depends on `output_tokens`, so on answer length, `max_new_tokens` and thinking.

`ask()` records the split as `first_token_s`. The cell runs a short answer and a long answer at three input sizes.

In [ ]:
rows = []
for budget in (256, 1024, 4096):
    for cap, kind in ((8, "short answer"), (200, "long answer")):
        r = ask("Describe this image in detail.", image=ROBOT_IMAGE, max_vision_tokens=budget, max_new_tokens=cap)
        rows.append({"call": f"{r['input_tokens']} in, {kind}", "input_tokens": r["input_tokens"], "output_tokens": r["output_tokens"],
                     "first_token_s": r["first_token_s"], "rest_s": round(r["gen_s"] - r["first_token_s"], 2), "gen_s": r["gen_s"]})
time_df = pd.DataFrame(rows)
display(time_df)

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.barh(time_df["call"], time_df["first_token_s"], color="#3b6ea5", label="before first token (reading the input)")
ax.barh(time_df["call"], time_df["rest_s"], left=time_df["first_token_s"], color="#b5532a", label="after first token (writing the answer)")
ax.invert_yaxis(); ax.set_xlabel("seconds"); ax.grid(axis="x", alpha=0.3)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False)
fig.tight_layout(); plt.show()

What to read here:

- `first_token_s` grows down the table with `input_tokens` and is the same for the short and the long answer at each input size.
- `rest_s` depends only on how much was written. For a long answer it is most of the call.
- So there are two different ways a call is slow. A short answer that is slow has too many input tokens: cut the vision budget or `fps`. A call that is slow because it writes a lot needs a shorter answer: ask for one word, a number or compact JSON, and keep thinking off unless you need it.

**Rule:** to speed up a call, first find which phase dominates. For a yes or no check on a camera frame it is the input. For captions and thinking it is the output.

## Part 9. Thinking mode: on by default, accuracy against time

Cosmos3-Edge can think step by step before it answers. Two facts about how that is controlled:

- **The switch is in the chat template, and it is on unless you turn it off.** Pass `enable_thinking=False` to `apply_chat_template()` for a direct answer. If you copy the minimal example from the model's documentation and leave the switch alone, every call thinks first, and a one-word question costs many times the tokens. In `ask()` the switch is `reasoning`, and it is off by default.
- **The output has no opening tag.** With thinking on, the prompt already ends with `<think>`, so the model writes its reasoning, then `</think>`, then the answer. `ask()` returns the part after `</think>` as `answer`.

NVIDIA's prompt guide also gives an instruction to add to the question, asking for the reasoning inside `<think>` tags followed by the final answer. The cell below runs one question four ways: thinking off, thinking on, thinking on with NVIDIA's instruction added (`reasoning="prompt"`), and the instruction alone with the switch off.

In [ ]:
SUM_QUESTION = "Large white digits flash on the screen one at a time. What is the sum of all the digits shown? Answer with the number only."
print("Correct answer:", sum(DIGITS), "\n")

print("--- thinking off ---")
show(ask(SUM_QUESTION, video=DIGIT_VIDEO, max_vision_tokens=2048, max_new_tokens=64))
print("\n--- thinking on ---")
show(ask(SUM_QUESTION, video=DIGIT_VIDEO, max_vision_tokens=2048, reasoning=True, max_new_tokens=800))
print("\n--- thinking on, with NVIDIA's format instruction ---")
show(ask(SUM_QUESTION, video=DIGIT_VIDEO, max_vision_tokens=2048, reasoning="prompt", max_new_tokens=800))
print("\n--- thinking off, NVIDIA's format instruction alone ---")
show(ask(SUM_QUESTION + "\n\n" + REASONING_PROMPT, video=DIGIT_VIDEO, max_vision_tokens=2048, max_new_tokens=800))

Compare `output_tokens` across the four. In the last one the switch is off and only the instruction asks for reasoning. Check whether the model thought at all: on this model the template switch decides, and the instruction by itself does not turn thinking on.

Now a small test set of five questions with known answers. Some need several steps (read, then compare or add). Some are single-step. Each is asked with thinking off, on, and on with NVIDIA's instruction.

In [ ]:
REASONING_QUESTIONS = [
    dict(video=DIGIT_VIDEO, max_vision_tokens=2048, q=SUM_QUESTION, truth="33"),
    dict(image=LABEL_IMAGE, q="Each box has a white label with a code. Which box number has the code whose last two digits form the largest number? Answer with the number only.", truth="5"),
    dict(video=DIGIT_VIDEO, max_vision_tokens=2048, q="Large white digits flash on the screen one at a time. Which digit appears immediately after the digit 9? Answer with the digit only.", truth="4"),
    dict(image=LABEL_IMAGE, q="Read the codes on the white labels. How many of the six codes contain the digit 8? Answer with the number only.", truth="4"),
    dict(image=ROBOT_IMAGE, q="Which robot hand is closer to the blue tape, the one on the left of the image or the one on the right? Answer left or right.",
         truth=lambda a: "right" in a.lower() and "left" not in a.lower()),
]

summaries = []
for reasoning in (False, True, "prompt"):
    summary, table = evaluate(REASONING_QUESTIONS, reasoning=reasoning)
    summaries.append({"reasoning": reasoning, **summary})
    print(f"\nreasoning = {reasoning!r}")
    display(table[["question", "truth", "answer", "correct", "output_tokens", "gen_s"]])
pd.DataFrame(summaries)

What to read here:

- **Time.** Compare `avg_output_tokens` and `total_gen_s` across the three rows. Thinking writes hundreds of tokens where a direct answer writes a handful, and time follows the tokens.
- **Accuracy.** Look at which questions flip from wrong to right. Multi-step questions (sum, compare) are where thinking has room to help. A question the model can answer at a glance gains nothing and can even go wrong after a long detour.
- **Following the format.** With thinking off, look for a direct answer that ignored "answer with the number only" and started explaining. It was cut at the token limit and scored wrong. Thinking gave the model somewhere to put the working, and the final answer came back short.
- **Cut-offs.** A row with no final answer is a thinking trace that ran past `max_new_tokens`. All that time was spent and nothing usable came back. This is why NVIDIA recommends 4096 or more output tokens with thinking. The cap here is 800 to keep the notebook quick on a small GPU.
- **Answers with no closing tag.** On an easy question the model sometimes skips the reasoning, writes the answer straight away and stops, without ever writing `</think>`. `ask()` treats that text as the answer. If you split on `</think>` yourself, handle the case where it is missing.
- Five questions is a demonstration, not a benchmark. Run your own questions before deciding (Part 18).

**Rule:** turn thinking off for simple perception and for anything latency-bound, and say so explicitly, because the default is on. Turn it on for questions that need steps, give it a generous `max_new_tokens`, and always check that a final answer came back.

## Part 10. Sampling: `temperature`, `top_p`, `top_k`, `repetition_penalty`, `seed`

At each step the model has a probability for every possible next token. The decoding settings decide how one is picked.

| Setting | Effect |
|---|---|
| `do_sample=False` | Greedy: always take the most likely token. Same input, same output. Everything above used this, so the experiments were repeatable. |
| `do_sample=True` | Draw the token at random according to the probabilities, shaped by the settings below. |
| `temperature` | Below 1 sharpens the distribution (safer, more repeatable). Above 1 flattens it (more varied, more errors). |
| `top_k` | Only the `k` most likely tokens may be drawn. |
| `top_p` | Only the smallest set of tokens whose probabilities add up to `p` may be drawn. |
| `repetition_penalty` | Above 1 discourages tokens already used. |
| `seed` | Fixes the random draw so a sampled run can be repeated. |

The model ships with one decoding setting: sampling is switched on. It does not ship a temperature, `top_p` or `top_k`, so if you call `generate()` and pass nothing, you get sampling at the library's fallback values, which are looser than the ones NVIDIA recommends. The next cell prints what the model carries.

In [ ]:
g = model.generation_config
print("shipped with the model:", {k: getattr(g, k) for k in ("do_sample", "temperature", "top_p", "top_k", "repetition_penalty")})
print("library fallback for a value shown as None: temperature 1.0, top_p 1.0, top_k 50, repetition_penalty 1.0")

In [ ]:
NEXT_STEP = "In one sentence, what should the robot do next?"
SETTINGS = {
    "greedy":                         dict(do_sample=False),
    "sampling, model defaults":       dict(do_sample=True),
    "sampling, NVIDIA's values":      dict(do_sample=True, temperature=0.7, top_p=0.8, top_k=20),
    "sampling, temperature 0.2":      dict(do_sample=True, temperature=0.2),
    "sampling, temperature 1.5, no top_k/top_p": dict(do_sample=True, temperature=1.5, top_k=0, top_p=1.0),
    "greedy, repetition_penalty 1.3": dict(do_sample=False, repetition_penalty=1.3),
}
rows = []
for name, kwargs in SETTINGS.items():
    for seed in (0, 1, 2):
        r = ask(NEXT_STEP, image=ROBOT_IMAGE, max_new_tokens=60, seed=seed, **kwargs)
        rows.append({"setting": name, "seed": seed, "gen_s": r["gen_s"], "answer": r["text"][:110]})
pd.DataFrame(rows)

What to read here:

- **Greedy** gives the same sentence for every seed.
- **Model defaults**, **NVIDIA's values** and **low temperature** stay close to the greedy answer, with wording that changes between seeds.
- **High temperature with `top_k` and `top_p` switched off** lets unlikely tokens through, and the answer can drift away from the question. `top_k` and `top_p` are the guard rails that make sampling usable.
- **`repetition_penalty`** changes the wording even under greedy decoding, because words get penalised once used. Here the answer comes out longer, and therefore slower.

Sampling settings barely move speed per token. They change time only by changing how long the answer gets.

The accuracy side matters most for tasks with one right answer. The next cell repeats the code-reading task from Part 5 five times per setting, at a vision budget (512) where greedy decoding reads every code. Watch how many runs still get all six once the pick is random.

In [ ]:
rows = []
for name, kwargs in [("greedy", dict(do_sample=False)),
                     ("sampling, model defaults", dict(do_sample=True)),
                     ("sampling, NVIDIA's values", dict(do_sample=True, temperature=0.7, top_p=0.8, top_k=20)),
                     ("sampling, temperature 1.3, no top_k/top_p", dict(do_sample=True, temperature=1.3, top_k=0, top_p=1.0))]:
    scores = []
    for seed in range(5):
        r = ask(READ_CODES, image=LABEL_IMAGE, max_vision_tokens=512, max_new_tokens=80, seed=seed, **kwargs)
        scores.append(codes_read(r["text"]))
    rows.append({"setting": name, "codes_correct_per_run (of 6)": scores, "mean_accuracy": round(sum(scores) / 30, 2)})
pd.DataFrame(rows)

Read the rows from top to bottom. With `top_k` and `top_p` in place the model keeps reading the codes it can see clearly. Take the guard rails away and raise the temperature, and a task it gets right every time becomes one it gets right most of the time. One wrong character in a part number is a wrong part number.

NVIDIA's documentation for the Cosmos 3 reasoner gives these values.

| Parameter | Without thinking | With thinking |
|---|---|---|
| `temperature` | 0.7 | 0.6 |
| `top_p` | 0.8 | 0.95 |
| `top_k` | 20 | 20 |
| `repetition_penalty` | 1.0 | 1.0 |
| `presence_penalty` | 1.5 | 0.0 |

`presence_penalty` exists on a vLLM server (Part 17). The `transformers` `generate()` call used in this notebook does not have it.

**Rule:** for extraction, measurement, JSON output and evaluation, use greedy decoding or a low temperature, and fix the seed. Use NVIDIA's sampling values when you want natural, varied text, such as several different captions for the same clip. Never rely on the values the model ships with: set them. There is one more reason to sample, and Part 14 shows it: with thinking on, greedy decoding can get stuck repeating itself.

## Part 11. Prompt structure: media order and the system prompt, tested

NVIDIA's guide gives two structural rules: put the media before the text, and keep the system prompt minimal (`You are a helpful assistant.`). Rules like these are cheap to test. The cell asks the same twelve questions under four structures and scores each one.

In [ ]:
STRUCTURE_QUESTIONS = [
    dict(video=DIGIT_VIDEO, max_vision_tokens=2048, q=LIST_DIGITS, truth=lambda a: digits_said(a) == DIGITS),
    dict(video=DIGIT_VIDEO, max_vision_tokens=2048, q="Large white digits flash on the screen one at a time. Which digit appears immediately after the digit 9? Answer with the digit only.", truth="4"),
    dict(image=LABEL_IMAGE, max_vision_tokens=512, q="Read the code printed on the white label of BOX 4. Answer with the code only.", truth="zr-7742"),
    dict(image=ROBOT_IMAGE, q="What color is the roll of tape? One word.", truth="blue"),
    dict(image=ROBOT_IMAGE, q="How many robot hands are visible? Answer with a number.", truth="2"),
    dict(image=ROBOT_IMAGE, q="Is the basket empty? Answer yes or no.", truth="yes"),
] + [dict(video=WAREHOUSE_VIDEO, fps=1, **item) for item in WAREHOUSE_QUESTIONS]

STRUCTURES = {
    "media first, default system prompt (NVIDIA's rule)": dict(),
    "text first, media after": dict(media_first=False),
    "no system prompt": dict(system=None),
    "long role-play system prompt": dict(system="You are a meticulous senior industrial safety inspector with twenty years of experience. "
                                                "Think about every detail of the scene and always justify your conclusions thoroughly."),
}
rows = []
for name, kwargs in STRUCTURES.items():
    summary, _ = evaluate(STRUCTURE_QUESTIONS, **kwargs)
    rows.append({"structure": name, **summary})
pd.DataFrame(rows)[["structure", "correct", "questions", "accuracy", "avg_output_tokens", "total_gen_s"]]

What to read here:

- Compare the `correct` column across the four rows. On a small set like this, differences of one question are noise. A structure only matters if it moves the score clearly.
- Compare `avg_output_tokens` on the role-play row with the others. A system prompt that asks for thoroughness pulls against a question that asks for one word. If the answers get longer, they cost time and can break a strict answer format.

This is the habit worth keeping from this part: when a guide tells you a prompt rule, test it on your own questions before you rely on it. Media first with the plain system prompt is still the right default, because it is what NVIDIA documents and it costs nothing.

## Part 12. Knowing when it is wrong

The model answers with the same fluent confidence whether it is right or wrong. Parts 5 and 6 showed that for blurred text and skipped frames. Two more checks help you catch errors without knowing the answer in advance.

### 12.1 Leading questions

A question that assumes something exists can push a model to play along. On this same test, Cosmos Reason 2 (2B) described the color of a coffee cup that is not in the picture. The cell asks about things that are not in the robot image, three ways: a neutral yes or no question, a leading question that presupposes the object, and the leading question with an escape clause added.

In [ ]:
ABSENT = [
    ("Is there a coffee cup on the table? Answer yes or no.", "What color is the coffee cup on the table?"),
    ("Is there a person's face visible in this image? Answer yes or no.", "What color is the hair of the person in the image?"),
    ("Is there a laptop on the table? Answer yes or no.", "What brand is the laptop on the table?"),
    ("Is there any fruit in the basket? Answer yes or no.", "What kind of fruit is in the basket?"),
]
ESCAPE = " If the thing I am asking about is not visible in the image, say that it is not visible."

rows = []
for neutral, leading in ABSENT:
    rows.append({"neutral question": ask(neutral, image=ROBOT_IMAGE, max_new_tokens=30)["text"][:40],
                 "leading question": leading,
                 "answer": ask(leading, image=ROBOT_IMAGE, max_new_tokens=30)["text"][:70],
                 "answer with escape clause": ask(leading + ESCAPE, image=ROBOT_IMAGE, max_new_tokens=30)["text"][:70]})
pd.DataFrame(rows)

None of these objects is in the image. Read across each row: the neutral question shows whether the model knows the object is absent, the leading question shows whether it invents one anyway, and the last column shows whether permission to say "not visible" changes anything. If the middle column already says the object is not there, this model resisted the trap on this image. Four questions on one picture do not prove it always will.

**Rule:** ask whether something is there before you ask about it, and give the model a way to say "not visible" or "cannot tell". It costs a few words, and it protects you on the images where the model would otherwise play along.

### 12.2 Answer agreement as a confidence signal

The model does not report a confidence score, but you can get one. Ask the same question several times with sampling switched on and count how often the answers agree. When the model has clearly read the answer from the picture, random sampling keeps landing on it. When it is guessing, the answers scatter.

The cell runs eight questions with known answers seven times each at temperature 1.0, and compares the agreement rate with whether the plain greedy answer was right.

In [ ]:
from collections import Counter

AGREEMENT_QUESTIONS = [
    dict(image=ROBOT_IMAGE, q="What color is the roll of tape? One word.", truth="blue"),
    dict(image=ROBOT_IMAGE, q="How many robot hands are visible? Answer with the number only.", truth="2"),
    dict(image=LABEL_IMAGE, max_vision_tokens=512, q="Read the code on the white label of BOX 4. Answer with the code only.", truth="zr-7742"),
    dict(image=LABEL_IMAGE, max_vision_tokens=96, q="Read the code on the white label of BOX 1. Answer with the code only.", truth="kx-4827"),
    dict(video=DIGIT_VIDEO, max_vision_tokens=2048, q="Large white digits flash on the screen one at a time. Which digit appears immediately after the digit 9? Answer with the digit only.", truth="4"),
    dict(video=DIGIT_VIDEO, max_vision_tokens=2048, q="Large white digits flash on the screen one at a time. What is the sum of all the digits shown? Answer with the number only.", truth="33"),
    dict(image=LABEL_IMAGE, q="Each box has a white label with a code. Which box number has the code whose last two digits form the largest number? Answer with the number only.", truth="5"),
    dict(image=LABEL_IMAGE, q="Read the codes on the white labels. How many of the six codes contain the digit 8? Answer with the number only.", truth="4"),
]

def agreement(item, runs=7):
    media = {k: v for k, v in item.items() if k not in ("q", "truth")}
    clean = lambda text: text.strip().lower().rstrip(".")
    greedy = clean(ask(item["q"], max_new_tokens=16, **media)["text"])
    samples = [clean(ask(item["q"], max_new_tokens=16, do_sample=True, temperature=1.0, top_p=1.0, top_k=0, seed=s, **media)["text"]) for s in range(runs)]
    top_answer, top_count = Counter(samples).most_common(1)[0]
    return {"question": item["q"][:55], "truth": item["truth"], "greedy_answer": greedy, "greedy_correct": is_correct(greedy, item["truth"]),
            "agreement": f"{top_count}/{runs}", "agreement_rate": round(top_count / runs, 2), "sampled_answers": ", ".join(samples)}

agree_df = pd.DataFrame([agreement(item) for item in AGREEMENT_QUESTIONS])
display(agree_df.drop(columns="agreement_rate"))
agree_df["agreement_level"] = np.where(agree_df["agreement_rate"] >= 0.85, "high (6 or 7 of 7)", "low (5 or fewer of 7)")
agree_df.groupby("agreement_level").agg(questions=("greedy_correct", "size"), greedy_correct=("greedy_correct", "sum"))

Read the bottom table: of the questions with high agreement, how many did the model get right, and how about the questions with low agreement? Scattered answers are a warning that the single answer you would normally take is unreliable. Agreement costs extra calls, and it is one of the few error signals you can get in production, where there is no answer key.

A caution: agreement measures consistency, and a model can be consistently wrong (the leading questions above). Treat low agreement as a red flag and high agreement as necessary, never as proof.

**Rule:** for decisions that matter, sample the question several times. Act on answers that agree, and route the ones that scatter to a person, a larger model, or a second look with more vision tokens or a tighter crop.

## Part 13. Real footage check: a warehouse camera

The test media so far was drawn to isolate one effect at a time. Real footage mixes everything: small people in a large frame, clutter, long duration. This part runs six factual questions on NVIDIA's 46 second warehouse clip. The answers were checked by eye against the footage: two people appear, the first wears a yellow hard hat and a high-visibility vest, the second wears neither, a box is dropped on the floor and the second person later picks it up, and the shelf posts carry the letters C, D, E and F.

In [ ]:
display(frame_strip(WAREHOUSE_VIDEO, n=6, height=120))
for item in WAREHOUSE_QUESTIONS:
    print("-", item["q"])
WAREHOUSE = [dict(video=WAREHOUSE_VIDEO, **item) for item in WAREHOUSE_QUESTIONS]

# every combination of three frame rates and three vision budgets
grid = []
for fps in (0.25, 1, 4):
    for budget in (256, 1024, 4096):
        summary, _ = evaluate(WAREHOUSE, fps=fps, max_vision_tokens=budget)
        grid.append({"fps": fps, "max_vision_tokens": budget, **summary})
grid_df = pd.DataFrame(grid)
print("Questions correct out of 6, for each fps (rows) and vision budget (columns):")
display(grid_df.pivot(index="fps", columns="max_vision_tokens", values="correct"))
print("Total generation seconds for the six questions:")
display(grid_df.pivot(index="fps", columns="max_vision_tokens", values="total_gen_s"))

On the test clips, the right `fps` and budget took the score from zero to perfect. Check whether that happens here. If the score stays in a narrow band across the whole grid, the limit is the model and the question, and no setting in that grid fixes it. A small model watching 46 seconds of cluttered footage in one pass has real limits.

The per-question view shows where it fails, direct and with thinking:

In [ ]:
for reasoning in (False, True):
    summary, table = evaluate(WAREHOUSE, fps=2, max_vision_tokens=4096, reasoning=reasoning)
    print(f"\nreasoning = {reasoning}: {summary['correct']}/{summary['questions']} correct, {summary['total_gen_s']} s")
    display(table[["question", "truth", "answer", "correct", "gen_s"]])

Compare the two tables. Thinking adds time. Check whether it raised the score, and whether any thinking run was cut off before it answered. On perception questions like these it often changes nothing, because the model is not short of thinking: it is short of seeing.

Two other things to try before giving up on a question.

**Ask the way NVIDIA asks.** NVIDIA's own safety prompt for this clip is an open question with thinking on. The same fact that the yes or no questions above probe (does everyone wear safety equipment?) is asked here in NVIDIA's words.

**Narrow the question to one moment.** `clip=` sends only the seconds where the second person appears, so the whole budget goes to them.

In [ ]:
r = ask("were all the workers working safety equipment", video=WAREHOUSE_VIDEO, fps=2, max_vision_tokens=4096, reasoning=True, max_new_tokens=1200)
print("NVIDIA's safety prompt, with reasoning. Final answer:\n")
print(r["answer"] or "(cut off while thinking)")
print(f"\noutput_tokens={r['output_tokens']}  gen_s={r['gen_s']}  hit_limit={r['hit_limit']}")

print("\nOne person at a time, using clip=(start, end):")
rows = []
for who, clip, hat_truth, vest_truth in (("first worker", (0, 12), "yes", "yes"), ("second person", (26, 45), "no", "no")):
    hat = ask("Is the person in this video wearing a hard hat? Answer yes or no.", video=WAREHOUSE_VIDEO, clip=clip, fps=2, max_vision_tokens=4096, max_new_tokens=8)["text"]
    vest = ask("Is the person in this video wearing a high-visibility vest? Answer yes or no.", video=WAREHOUSE_VIDEO, clip=clip, fps=2, max_vision_tokens=4096, max_new_tokens=8)["text"]
    rows.append({"who": who, "clip_seconds": clip, "hard hat (truth)": hat_truth, "hard hat (model)": hat,
                 "vest (truth)": vest_truth, "vest (model)": vest})
pd.DataFrame(rows)

Compare the open answer with what is in the footage: the second person wears neither a hard hat nor a vest. Then read the table, model against truth, for each person. A high-visibility vest hangs on a post beside the pallet for the whole clip, and hard hats and vests are in the same frames as a person who wears neither. A wrong "yes" there is the kind of error that only shows up on real footage, and it is why a test set built from your own camera is worth more than any published benchmark.

**What this part is for:** settings matter most when the model *can* do the task and the input is starving it. When the score does not move with the settings, change the question, narrow the clip, add thinking, or use a larger model (Cosmos 3 also comes as Nano and Super).

## Part 14. Task gallery: NVIDIA's prompts on NVIDIA's footage

The reasoner was trained for specific physical AI tasks, and each has a prompt form that works best. These are prompts from NVIDIA's Cosmos Cookbook prompt guide, run on the media that guide uses. NVIDIA's prompt guide for the Cosmos 3 reasoner uses the same forms. Thinking is on where the guide turns it on.

In [ ]:
GALLERY = [
    ("Robot: next action", dict(video=PACKING_VIDEO), "What can be the next immediate action?", True),
    ("Robot: goal verification", dict(video=PACKING_VIDEO), "Has the robot successfully completed the intended packaging subtask? Explain why not.", True),
    ("Driving: next action", dict(video=DRIVING_VIDEO, max_vision_tokens=4096), "What's the next immediate action for the Ego vehicle?", True),
    ("Video critic (is this generated clip physically plausible?)", dict(video=GENERATED_VIDEO, max_vision_tokens=4096),
     "Approve or reject this generated video for inclusion in a dataset for physical world model ai training. It must perfectly adhere to physics, "
     "object permanence, and have no anomalies. Any issue or concern causes rejection.", True),
    ("Security", dict(video=DOORSTEP_VIDEO, max_vision_tokens=4096), "Is the person allowed to pick up the packages?", True),
    ("Timestamped events (JSON)", dict(video=DOORSTEP_VIDEO, max_vision_tokens=4096),
     "Describe the notable events in the provided video. Provide the result in json format with 'mm:ss.ff' format for time depiction for each event. "
     "Use keywords 'start', 'end' and 'caption' in the json output.", False),
]
gallery_rows = []
for title, media, prompt, reasoning in GALLERY:
    r = ask(prompt, reasoning=reasoning, max_new_tokens=1000, **media)
    print("=" * 100, f"\n{title}\nPROMPT: {prompt}" + ("   [+ reasoning]" if reasoning else ""))
    display(frame_strip(media["video"], n=5, height=100))
    print(r["answer"] or "(no final answer: cut off while thinking)")
    gallery_rows.append({"task": title, "reasoning": reasoning, "frames": r["frames"], "input_tokens": r["input_tokens"],
                         "output_tokens": r["output_tokens"], "gen_s": r["gen_s"], "hit_limit": r["hit_limit"]})
pd.DataFrame(gallery_rows)

Look at the `hit_limit` column. If the video critic row reached the limit, it never gave a verdict. The next cell shows why, and what fixes it. It prints the end of what the model wrote under greedy decoding, then runs the same call three times with NVIDIA's sampling values for thinking.

In [ ]:
title, media, prompt, _ = GALLERY[3]                      # the video critic
greedy = ask(prompt, reasoning=True, max_new_tokens=1000, **media)
print("Greedy decoding. hit_limit =", greedy["hit_limit"], "  The last 150 characters it wrote:\n", greedy["text"][-150:])

rows = []
for seed in (0, 1, 2):
    r = ask(prompt, reasoning=True, max_new_tokens=1000, do_sample=True, temperature=0.6, top_p=0.95, top_k=20, seed=seed, **media)
    rows.append({"decoding": "NVIDIA's values for reasoning", "seed": seed, "output_tokens": r["output_tokens"],
                 "hit_limit": r["hit_limit"], "gen_s": r["gen_s"], "final answer": r["answer"][:80]})
pd.DataFrame(rows)

If the greedy run ended in the same sentence written over and over, the model was stuck in a loop: under greedy decoding, once a repeated sentence becomes the most likely continuation, it stays the most likely. Sampling breaks the loop, because it does not always take the top token. This is the reason NVIDIA's values for thinking include a temperature. The price is in the table: the verdict can differ from seed to seed.

**Rule:** when a thinking call hits the limit, read the end of the text before you raise the limit. If it repeats, a higher limit only buys a longer loop. Switch that call to NVIDIA's sampling values, fix the seed, and for a decision that matters run several seeds and compare (Part 12).

Check each gallery answer against its frames. In the driving clip a worker holds up a STOP sign in the construction zone. In the generated clip a second blue block appears from nowhere, which is the kind of flaw the critic prompt exists to catch. The security question has no fact to check: it asks for a judgement, and the model's reasoning shows what it assumed. Treat judgement answers as a starting point for a person, never as a ruling.

### 14.1 Bounding boxes (2D grounding)

The prompt is `Locate the bounding box of {object}. Return a json.` The box comes back as `[x1, y1, x2, y2]` on a 0 to 1000 grid, whatever the image size, so multiply by `width / 1000` and `height / 1000` to get pixels.

In [ ]:
def draw_boxes(image_path, objects, colors=("red", "lime", "cyan", "yellow")):
    picture = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(picture)
    W, H = picture.size
    for obj, color in zip(objects, colors):
        r = ask(f"Locate the bounding box of {obj}. Return a json.", image=image_path, max_new_tokens=128)
        print(r["text"], "\n")
        try:
            for item in parse_json(r["text"]):
                x1, y1, x2, y2 = item["bbox_2d"]
                draw.rectangle([x1 * W / 1000, y1 * H / 1000, x2 * W / 1000, y2 * H / 1000], outline=color, width=max(3, W // 250))
        except Exception as e:
            print(f"Could not parse a box for {obj}: {e}")
    return picture

display(draw_boxes(ROBOT_IMAGE, ["the blue tape", "the basket"]).resize((640, 400)))
display(draw_boxes(FORKLIFT_IMAGE, ["the pallet of water bottles", "the forklift driver"]))

Name the object by what is visible ("the pallet of water bottles") and check the drawn box against the picture. The `try` block above is there because a grounding call can come back as text that does not parse.

### 14.2 A robot trajectory (action chain of thought)

The model can also return points: a path for a gripper to follow, on the same 0 to 1000 grid. This is NVIDIA's "Action CoT" prompt on the robot image.

In [ ]:
TRAJECTORY = ('You are given the task "Move the tape into the basket". Specify the 2D trajectory your end effector should follow in pixel space. '
              'Return the trajectory coordinates in JSON format like this: {"point_2d": [x, y], "label": "gripper trajectory"}.')
r = ask(TRAJECTORY, image=ROBOT_IMAGE, reasoning=True, max_new_tokens=1000)
print(r["answer"] or "(no final answer: cut off while thinking)")

picture = Image.open(ROBOT_IMAGE).convert("RGB")
draw = ImageDraw.Draw(picture)
W, H = picture.size
try:
    points = [(p["point_2d"][0] * W / 1000, p["point_2d"][1] * H / 1000) for p in parse_json(r["answer"])]
    draw.line(points, fill="red", width=8)
    for i, (x, y) in enumerate(points):
        draw.ellipse([x - 16, y - 16, x + 16, y + 16], fill="yellow" if i == 0 else "red", outline="black", width=3)
    print("\nYellow dot: start. The path should reach the tape, then end over the basket.")
except Exception as e:
    print("Could not parse the trajectory:", e)
display(picture.resize((640, 400)))

Compare the drawn path with the task: it should reach the tape and end over the basket. A list with the same point several times, or labels that do not match the points, is the model being unsure. Check `hit_limit` on JSON calls in particular. A cut-off JSON answer does not parse.

## Part 15. Several images, and follow-up questions

### 15.1 More than one image in a call

`ask()` accepts a list of images. Each is encoded separately and they appear in the prompt in order, so the model can compare them. `max_vision_tokens` applies per image. The test: two copies of the label image, with one character changed on one box in the second.

In [ ]:
changed_codes = list(CODES); changed_codes[4] = "HD-5297"        # BOX 5: last digit 1 -> 7
CHANGED_IMAGE = make_label_image("labels_changed.png", changed_codes)

r = ask("These two images show the same six boxes at two different times. The code on one box label is different in the second image. "
        "Which box number changed, and what are the old and new codes? Answer in one sentence.",
        image=[LABEL_IMAGE, CHANGED_IMAGE], max_vision_tokens=1024, max_new_tokens=80)
print("Truth: BOX 5 changed from HD-5291 to HD-5297\n")
show(r)

`input_tokens` is about double a single-image call. Two images cost two budgets. Check the answer in two steps: did it name the right box, and did it get old and new the right way round?

### 15.2 Follow-up questions (multi-turn chat)

A conversation is a growing list of messages. To ask a follow-up, append the model's answer as an `assistant` message and your next question as a new `user` message, then send the whole list again. The image is sent once, in the first turn, and its tokens are re-read on every turn.

In [ ]:
class Chat:
    def __init__(self, image, max_vision_tokens=1024):
        self.image = (Image.open(image) if isinstance(image, str) else image).convert("RGB")
        self.size = {"shortest_edge": 16 * PIXELS_PER_TOKEN, "longest_edge": max_vision_tokens * PIXELS_PER_TOKEN}
        self.messages = [{"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]}]

    def say(self, text, max_new_tokens=120):
        content = [{"type": "text", "text": text}]
        if len(self.messages) == 1:                       # first turn: the image goes before the text
            content.insert(0, {"type": "image"})
        self.messages.append({"role": "user", "content": content})
        prompt_text = processor.apply_chat_template(self.messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = processor(text=[prompt_text], images=[self.image], return_tensors="pt", size=self.size).to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        answer = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        self.messages.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
        print(f"YOU:   {text}\nMODEL: {answer}\n       (input tokens this turn: {inputs['input_ids'].shape[1]})\n")

chat = Chat(ROBOT_IMAGE)
chat.say("What objects are on the table? Answer in one sentence.")
chat.say("Which of those is closest to the bottom of the image?")
chat.say("What color is it?")

The follow-ups say "those" and "it", and the model resolves them from the earlier turns. The input token count rises each turn, because the whole history is sent again. Long conversations fill the context window (Part 7), so drop old turns when they stop mattering.

## Part 16. Speed and memory: batching, 16-bit formats and 4-bit loading

### 16.1 Batching: several questions in one pass

A GPU can process several prompts side by side. Instead of eight calls, build eight prompts, pad them to the same length and run `generate` once. Padding must go on the left, so every prompt ends right where the answer begins.

In [ ]:
BATCH_QUESTIONS = ["What color is the roll of tape? One word.", "How many robot hands are visible? Answer with a number.",
                   "Is the basket empty? Answer yes or no.", "What is the table made of? One word.",
                   "Is there a person in the image? Answer yes or no.", "What shape is the basket? One word.",
                   "What color is the basket? One word.", "What color is the curtain behind the table? One word."]

def ask_batch(prompts, image, max_vision_tokens=1024, max_new_tokens=12):
    picture = (Image.open(image) if isinstance(image, str) else image).convert("RGB")
    texts = [processor.apply_chat_template(
        [{"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
         {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": p}]}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False) for p in prompts]
    processor.tokenizer.padding_side = "left"
    size = {"shortest_edge": 16 * PIXELS_PER_TOKEN, "longest_edge": max_vision_tokens * PIXELS_PER_TOKEN}
    inputs = processor(text=texts, images=[picture] * len(prompts), return_tensors="pt", padding=True, size=size).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return [t.strip() for t in processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)]

torch.cuda.synchronize(); t0 = time.perf_counter()
one_by_one = [ask(q, image=ROBOT_IMAGE, max_new_tokens=12)["text"] for q in BATCH_QUESTIONS]
torch.cuda.synchronize(); t1 = time.perf_counter()
torch.cuda.reset_peak_memory_stats()
batched = ask_batch(BATCH_QUESTIONS, ROBOT_IMAGE)
torch.cuda.synchronize(); t2 = time.perf_counter()

display(pd.DataFrame({"question": BATCH_QUESTIONS, "one by one": one_by_one, "batched": batched}))
print(f"one by one: {t1 - t0:.2f} s    batched: {t2 - t1:.2f} s    speed-up: {(t1 - t0) / (t2 - t1):.1f}x    "
      f"peak GPU memory while batched: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

What to read here: the answers should match between the two columns, the batched run should finish sooner, and its peak memory is higher, because eight copies of the image are in memory at once. How much time you save depends on the GPU: a batch helps most when single calls leave the GPU partly idle. Batch size is limited by memory, so lower the vision budget if a large batch runs out.

This helper sends the same image eight times. For different images per question, pass a list of images in the same order as the prompts.

### 16.2 Weight formats: float16, bfloat16 and 4-bit

The weights loaded in Part 1 are float16. Two other ways to load the same model are worth knowing about:

- **bfloat16** is the other 16-bit format, and the only one NVIDIA says it tested. It takes the same memory.
- **4-bit** loading through `bitsandbytes` stores the weights in 4 bits, which cuts the memory the model holds to under half. NVIDIA does not list it as supported for this model. It worked on Cosmos Reason 2 (2B) at a small cost in accuracy, so it is worth knowing what it does here.

The cell loads each of the two as a second copy, one at a time, and runs the same tests on all three: memory, speed, reading the codes, the digit clip, and the first words of a description.

In [ ]:
import gc
from transformers import BitsAndBytesConfig

def measure(name, m, weights_gb):
    _ = ask("Hi", image=ROBOT_IMAGE, max_new_tokens=4, use_model=m)     # warm-up
    codes = {b: codes_read(ask(READ_CODES, image=LABEL_IMAGE, max_vision_tokens=b, max_new_tokens=80, use_model=m)["text"]) for b in (256, 512, 1024)}
    digits = ask(LIST_DIGITS, video=DIGIT_VIDEO, max_vision_tokens=2048, max_new_tokens=40, use_model=m)
    speed = ask("Describe this image in detail.", image=ROBOT_IMAGE, max_new_tokens=200, use_model=m)
    return {"weights": name, "weights_gb": round(weights_gb, 1), "peak_gpu_gb": speed["peak_gpu_gb"], "tokens_per_s": speed["tokens_per_s"],
            "codes read at 256 / 512 / 1024 tokens (of 6)": f"{codes[256]} / {codes[512]} / {codes[1024]}",
            "digit video correct": digits_said(digits["text"]) == DIGITS, "start of a description": speed["text"][:60]}

held = torch.cuda.memory_allocated()
rows = [measure("float16 (loaded in Part 1)", model, held / 1e9)]
for name, load_kwargs in (
    ("bfloat16", dict(dtype=torch.bfloat16)),
    ("4-bit (bitsandbytes nf4)", dict(quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16))),
):
    other = transformers.AutoModelForImageTextToText.from_pretrained(MODEL_ID, device_map="auto", **load_kwargs)
    rows.append(measure(name, other, (torch.cuda.memory_allocated() - held) / 1e9))
    del other; gc.collect(); torch.cuda.empty_cache()      # free this copy before loading the next
display(pd.DataFrame(rows))

What to read here:

- **float16 against bfloat16.** Compare the two rows column by column. If they match, the format you can run is the format to use.
- **4-bit.** `weights_gb` drops, and then read the last three columns. If no codes are read, the digit clip fails and the description is not words, this way of loading does not work for this model. The load itself raises no error and no warning, so nothing tells you except the output.

**Rule:** a smaller format is only a saving if the answers survive it. Run a test with a known answer straight after you change how a model is loaded, before you measure anything else. For this model, stay with a 16-bit format in `transformers`. NVIDIA publishes its own smaller version for serving, [Cosmos3-Edge-FP8](https://huggingface.co/nvidia/Cosmos3-Edge-FP8), which this notebook does not test.

## Part 17. Streaming, and the same parameters on a server

### 17.1 Streaming

Generation takes the same total time either way, but with a streamer you see tokens as they are written. For anything a person waits on, the time to the first word (Part 8) matters more than the total.

In [ ]:
from transformers import TextStreamer
streamer = TextStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)
r = ask("Describe this image in three sentences.", image=ROBOT_IMAGE, max_new_tokens=150, streamer=streamer)

### 17.2 The same parameters on a vLLM server

This notebook calls the model in-process with `transformers`, which is the simplest way to learn. For deployment NVIDIA documents serving it with vLLM, which exposes an OpenAI-compatible API. The ideas are identical and the names change.

Nothing in this section runs in the notebook, because vLLM installs its own PyTorch and belongs in a separate environment. Every request below was run against a live server with **vLLM 0.33.0** on an RTX 5090 workstation (not on Colab). vLLM changes quickly, so check the flags against your version. NVIDIA also publishes a container for this model, `vllm/vllm-openai:cosmos3`.

**Start the server.** This is NVIDIA's command from the model card plus one flag, `--trust-request-mm-kwargs`, which current vLLM needs before it accepts per-request image and video settings. Only add that flag when you trust the clients, because those settings can raise the load on the server.

```bash
vllm serve nvidia/Cosmos3-Edge \
  --max-model-len 131072 \
  --mm-processor-kwargs '{"do_resize": true, "min_pixels": 4096, "max_pixels": 16777216}' \
  --media-io-kwargs '{"video": {"num_frames": -1}}' \
  --trust-request-mm-kwargs \
  --port 8000
```

For the tests below I started it with `--max-model-len 8192`, to see what happens at the limit.

**Call it** with the OpenAI client:

```python
import base64
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

def data_url(path, mime):
    return f"data:{mime};base64," + base64.b64encode(open(path, "rb").read()).decode()

response = client.chat.completions.create(
    model="nvidia/Cosmos3-Edge",
    messages=[
        {"role": "system", "content": [{"type": "text", "text": "You are a helpful assistant."}]},
        {"role": "user", "content": [
            {"type": "video_url", "video_url": {"url": data_url("digits.mp4", "video/mp4")}},   # media first
            {"type": "text", "text": "List every digit that appears, in order."},
        ]},
    ],
    max_tokens=40,                 # max_new_tokens
    temperature=0,                 # greedy
    extra_body={
        "media_io_kwargs": {"video": {"fps": 4, "num_frames": -1}},      # fps for this request
        "mm_processor_kwargs": {"max_pixels": 4096 * 1024},              # vision token budget, in pixels (tokens x 1024)
        "chat_template_kwargs": {"enable_thinking": False},              # thinking off
    },
)
print(response.choices[0].message.content, response.usage)
```

| In this notebook | On a vLLM server | Observed in testing |
|---|---|---|
| context window (Part 7) | `--max-model-len` at server start | A request with a longer input is refused: `Input length (10782) exceeds model's maximum context length (8192)`. A `max_tokens` above the limit is refused too: `max_tokens=9000 cannot be greater than max_model_len=8192`. |
| `max_new_tokens` | `max_tokens` in the request | |
| `fps` | `extra_body={"media_io_kwargs": {"video": {"fps": 4}}}` | Refused unless the server is started with `--trust-request-mm-kwargs`. On the digit clip, 1 fps returned three of the six digits. Putting `fps` in `mm_processor_kwargs` failed with a processor error. |
| `max_vision_tokens` | `extra_body={"mm_processor_kwargs": {"max_pixels": N}}` in pixels (tokens x 1024) | Needs the same flag. The robot image cost 2,254 prompt tokens at the server's default and 274 with `max_pixels` set to 256 x 1024. For video the number covers the whole clip: the digit clip at 4 fps was refused at the default (10,782 tokens) and returned all six digits in 4,390 tokens with `max_pixels` set to 4096 x 1024. |
| `reasoning=True` | nothing to send: thinking is on by default | The thinking and the answer arrive together in `message.content`, as reasoning, then `</think>`, then the answer. There is no opening tag. Split on `</think>` yourself. |
| thinking off (the default in `ask()`) | `extra_body={"chat_template_kwargs": {"enable_thinking": False}}` | The same one-word question cost 30 output tokens with nothing set and 2 with thinking off. |
| `temperature`, `top_p`, `seed` | same names in the request | |
| `top_k`, `repetition_penalty` | inside `extra_body` | |
| (not available here) | `presence_penalty` in the request | NVIDIA's value for calls without thinking is 1.5. |

Two differences to expect. The server's default image size is the full picture, where `ask()` caps it at 1,024 tokens, so the same file costs more tokens there until you set `max_pixels`. And every request thinks unless you switch it off. Read `response.usage` on your first calls: it shows both.

## Part 18. Tune it on your own data

Everything above used someone else's media. The method transfers directly:

1. Collect a few images or clips from **your** camera.
2. Write 10 to 20 questions whose answers you have checked by eye. Include the cases you care most about getting right.
3. Sweep one setting at a time with `sweep()` and pick the cheapest value where accuracy has levelled off.

The cell below is a template. It runs on the warehouse clip so it works as it stands. Replace `MY_QUESTIONS` with your own.

In [ ]:
MY_QUESTIONS = [
    # media                                 question                                                  checked answer
    dict(video=WAREHOUSE_VIDEO, clip=(0, 12),  q="Is the person in this video wearing a hard hat? Answer yes or no.", truth="yes"),
    dict(video=WAREHOUSE_VIDEO, clip=(0, 12),  q="What is the person carrying? One word.",                             truth="box"),
    dict(video=WAREHOUSE_VIDEO, clip=(26, 45), q="Is the person in this video wearing a hard hat? Answer yes or no.", truth="no"),
    dict(video=WAREHOUSE_VIDEO, clip=(26, 45), q="Does the person pick something up from the floor? Answer yes or no.", truth="yes"),
    dict(image=FORKLIFT_IMAGE,                 q="What color is the forklift? One word.",                             truth="yellow"),
    dict(image=FORKLIFT_IMAGE,                 q="Is the forklift carrying a load? Answer yes or no.",                 truth="yes"),
    dict(image=ROBOT_IMAGE,                    q="Is the basket empty? Answer yes or no.",                             truth="yes"),
    dict(image=LABEL_IMAGE,                    q="Read the code on the white label of BOX 2. Answer with the code only.", truth="bt-9153"),
]

budget_sweep = sweep(MY_QUESTIONS, "max_vision_tokens", [64, 256, 1024, 4096], fps=2)
display(budget_sweep)
two_panel(budget_sweep, "max_vision_tokens", "Your questions: vision token budget")

summary, table = evaluate(MY_QUESTIONS, fps=2, max_vision_tokens=1024)
display(table[["question", "truth", "answer", "correct", "gen_s"]])

Read the sweep from the top: find the first row where `accuracy` stops improving, and use that value. Everything above it is time and memory with nothing in return. Then read the per-question table and look at what is still wrong. Those are the cases to fix with a crop, a clip, reasoning or a larger model, in that order of cost.

Other sweeps to try with the same call:

```python
sweep(MY_QUESTIONS, "fps", [0.5, 1, 2, 4], max_vision_tokens=1024)        # video questions only
sweep(MY_QUESTIONS, "reasoning", [False, True], max_vision_tokens=1024)
sweep(MY_QUESTIONS, "media_first", [True, False])
```

To use your own files, upload them with the folder icon in Colab's left bar, then:

```python
show(ask("What is happening here?", image="my_photo.jpg"))
show(ask("Caption the video in detail.", video="my_clip.mp4", fps=4, max_vision_tokens=2048))
show(ask("What happens here?", video="my_clip.mp4", clip=(10, 20)))       # only seconds 10 to 20
```

## Part 19. Cheat sheet

| Parameter | What it controls | Raise it when | Cost of raising it | Start with |
|---|---|---|---|---|
| `max_new_tokens` | longest possible answer | answers are cut off (`hit_limit`) | none, unless the model rambles | 256 for answers, 4096 with thinking |
| `max_vision_tokens` | image or clip detail | small text or small objects are misread | time to first token, memory | 1024 for images; for video, about 32 per frame or more |
| `fps` | how often the model looks at a video | short events are missed | more frames, less detail per frame | 4 (NVIDIA's recommended value) |
| `clip` | which seconds of a video are sent | (narrow it for one moment) | more calls, errors at clip boundaries | whole video |
| `reasoning` (`enable_thinking`) | think before answering | the question needs several steps | many times the output tokens and time | off, and say so: the model's own default is on |
| `do_sample`, `temperature` | randomness | you want varied wording, or a thinking call loops | repeatability, and accuracy at high values | greedy for anything you measure |
| `top_p`, `top_k` | which tokens sampling may pick | (lower them to make sampling safer) | less variety | NVIDIA's values: 0.8 and 20. The model ships none. |
| `repetition_penalty` | reuse of tokens | the output loops | can distort exact strings such as codes | 1.0 |
| `seed` | the random draw | never: fix it | none | any fixed number |
| context window | input plus output tokens | you send long video | memory | count first with `build_inputs()` |

**When a call is slow**

| What you see | Cause | Fix |
|---|---|---|
| `first_token_s` is most of `gen_s` | too many input tokens | lower `max_vision_tokens` or `fps`, crop, or clip |
| `first_token_s` is small, `gen_s` is large | long output | ask for a shorter answer format, turn thinking off |
| a one-word question takes seconds | thinking is on (the model's default) | pass `enable_thinking=False` (Part 9) |
| many short calls | per-call overhead | batch them (Part 16) |

**When an answer is wrong**

| What you see | Likely cause | Fix |
|---|---|---|
| wrong small details, stated confidently | vision budget too low | raise it, or crop (Part 5) |
| a short event is missing | `fps` too low | raise `fps` (Part 6) |
| a made-up sequence from a video | too few tokens per frame | raise the budget or lower `fps` (Part 6) |
| events doubled or lost in a split video | clip boundaries | fewer clips, or overlap and remove duplicates (Part 6.3) |
| wrong on a multi-step question | no room to work it out | thinking on (Part 9) |
| a thinking call runs to the limit repeating itself | greedy decoding stuck in a loop | NVIDIA's sampling values for thinking (Part 14) |
| describes something that is not there | leading question | ask if it exists first, add an escape clause (Part 12) |
| different answer each run | sampling is on, or the model is guessing | greedy decoding. If sampled answers scatter, treat the answer as unreliable (Part 12) |
| still wrong at every setting | beyond the model | change the question, or use a larger model (Part 13) |

### Troubleshooting

| Symptom | Cause and fix |
|---|---|
| The download is slow or stops with a rate limit message | Add an `HF_TOKEN` secret with notebook access (Before you start, step 3). |
| An error that the model type `cosmos3_edge` is not recognized | The `transformers` release is too old. Run the install cell in 1.2, then **Runtime > Restart session**. |
| `CUDA out of memory` | Lower `max_vision_tokens` or `fps`, or shorten the clip. Then **Runtime > Restart session** to clear the GPU. |
| Answer stops mid-sentence, or thinking has no final answer | `hit_limit` is `True`. Read the end of the text. If it repeats, see Part 14. If not, raise `max_new_tokens`. |
| JSON does not parse | The answer was cut off (`hit_limit`), or thinking text came before it. Parse the `answer` field that `ask()` returns. |
| Output is symbols, not words | The model was loaded in 4-bit. Load it in a 16-bit format (Part 16). |
| Video loading errors about `torchcodec` or `torchvision` | The library's built-in video loader is in use. Read frames with OpenCV as `read_video()` does here. |
| A media file failed to download | The part that uses it fails and the others still run. Re-run the cell in 1.5. |
| A vLLM answer starts with the model talking to itself and contains `</think>` | Thinking is on by default. Send `enable_thinking: False`, or split on `</think>`. See Part 17. |

### Sources

- Model card: https://huggingface.co/nvidia/Cosmos3-Edge
- NVIDIA Cosmos, code, cookbook and model reference (sizes, sampling values): https://github.com/NVIDIA/cosmos
- Cosmos Framework, with the reasoner prompt guide and the Transformers and vLLM references: https://github.com/NVIDIA/cosmos-framework
- The model in the `transformers` documentation: https://huggingface.co/docs/transformers/model_doc/cosmos3_edge
- Sample footage and task prompts (Cosmos Cookbook prompt guide): https://nvidia-cosmos.github.io/cosmos-cookbook/getting_started/prompt_guide/reason_guide.html
- The same tests on Cosmos Reason 2 (2B): https://github.com/obiedeh/cosmos-reason2-first-timers-guide
- vLLM: https://docs.vllm.ai

### About the author

Obinna Edeh runs EmbodiedEdge, a Physical AI lab. He designs and builds Physical AI and edge AI systems, and proves them on real robots and hardware. He runs Cosmos Reason 2 and Cosmos3-Edge on a Jetson AGX Thor.

- Urban Edge Vision Analytics: https://github.com/obiedeh/urban-edge-vision-analytics ([video](https://www.youtube.com/watch?v=PtIh9hnD8Lo))
- Physical AI Safety Observability: https://github.com/obiedeh/physical-ai-safety-observability ([video](https://www.youtube.com/watch?v=bXhttNSlDP0))
- GitHub: https://github.com/obiedeh
- LinkedIn: https://www.linkedin.com/in/obinna-edeh-206306137/
- Lab: https://embodiededge.ai

This is independent work. It is not affiliated with or endorsed by NVIDIA. Tested on Google Colab T4 and A100 runtimes and on an RTX 5090 workstation.